# Joint 6-D Diffusion for Colored Point Cloud Completion

**System A** of a two-system comparison. Trains a single diffusion model over
(x, y, z, r, g, b) **jointly** to complete occluded colored point clouds.

System B (`Pipeline_RePaintPart_kaggle.ipynb`) runs the existing frozen-PoinTr → PointNet part-seg
→ RePaint-part pipeline on **exactly the same objects with exactly the same metric code**, and
`Comparison_Report_kaggle.ipynb` merges the two. The shared spec below is what makes that merge
legitimate rather than decorative.

## Why this design — every choice traces to a measured result

| finding | measured | consequence here |
|---|---|---|
| **kNN-aggregated geometry conditioning is what works** | eps-MSE **-63.2 %**, win **40/40**, p<1e-4 | EdgeConv neighbourhoods are the backbone; the clean partial enters through a **cross-neighbourhood** path |
| **Pointwise geometry conditioning does nothing** | eps -1.9 %, win 45 %, CI spans 0 | kept only as the control that makes the above claim testable |
| **Part masking is harmful** | eps -6.0 %, boundary ΔE00 -4.1 %, win 12.5 % | **no part labels anywhere** — the segmentation stage and its error source are gone |
| **Part identity is neutral** | all metrics unresolved | not worth its complexity either |
| **Colour helps geometry structurally** | Chamfer -26.1 % on 40/40 | justifies diffusing all 6 channels jointly |

**The capacity confound is closed by construction:** the -63.2 % result came from a model with
+66.8 % more parameters, which is why it could never be claimed outright. Here every variant is
width-tuned to the same budget (§8).

## §1 · Configuration

In [ ]:
# ======================= SHARED — must match System B =========================
SYSTEM_NAME   = "joint6d"
CATEGORY      = "airplane"        # airplane | car | chair
DIFFICULTIES  = ["moderate"]      # subset of ["simple","moderate","hard"] = 25/50/75% occluded
NUM_POINTS    = 2048              # points per cloud, subsampled from the 8192-pt GT
NUM_TRAIN     = 150
NUM_VAL       = 40
SEED          = 0
FSCORE_TAU    = 0.01              # F-score threshold, partial-frame units
MATCH_TAU     = 0.02              # colour scored only on GT points matched within this radius
EQUALIZE_PRED_N = "gt"            # "gt" | int | None — see spec_score(); the density control
EVAL_REPEATS  = 3                 # independent sampling runs per object
# ==============================================================================

# ------------------- SYSTEM A ONLY (nothing below affects System B) -----------
TRAIN_MODELS  = ["J6D-pw", "J6D-self", "J6D"]
# J6D-pw   : global max-pool only         (the "no local aggregation" control)
# J6D-self : + EdgeConv over the NOISY cloud
# J6D      : + cross-neighbourhood aggregation from the CLEAN partial   <- the A2 mechanism
PARAM_TARGET  = 1_200_000
PARAM_TOL     = 0.02
N_BLOCKS      = 4
KNN_SELF      = 16
KNN_CROSS     = 16

T_STEPS       = 1000
SCHEDULE      = "cosine"
SAMPLER       = "ddim"            # "ddim" (fast) or "ddpm"
DDIM_STEPS    = 100
REPAINT       = True
REPAINT_JUMP  = 10                # jumps multiply SAMPLING cost ~3x — §11 counts it exactly
REPAINT_NJUMP = 1

NUM_EPOCHS    = 600
BATCH_SIZE    = 8
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
VAL_EVERY     = 5
GRAD_CLIP     = 1.0
EMA_DECAY     = 0.999
N_BOOT        = 2000

RUN_SANITY_CHECK    = True
RUN_FULL_EXPERIMENT = False       # <- flip to True after the sanity check passes
SANITY_STEPS        = 800

PILOT = False
# PILOT=True runs the ENTIRE pipeline end to end — train, sample, score, plot, report — in a few
# minutes. The §11 sanity check only proves the models can learn; it never touches the sampler,
# the metrics, or any figure, which is where a long run actually dies. Do one pilot pass, confirm
# a report is written, then set PILOT=False.
# ==============================================================================

if PILOT:
    NUM_EPOCHS, EVAL_REPEATS, DDIM_STEPS = 20, 1, 25
    NUM_VAL, NUM_TRAIN, SANITY_STEPS = 8, 24, 100
    RUN_FULL_EXPERIMENT = True     # forced: a pilot that stops at the sanity check tests nothing
    print(">>> PILOT MODE — results are meaningless; this only proves the plumbing works.")
    print(">>> RUN_FULL_EXPERIMENT forced True so sampling, metrics and every figure execute.\n")

import os, sys, math, json, time, warnings, hashlib, random
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)

ON_KAGGLE   = os.path.isdir("/kaggle")
RESULTS_DIR = Path("/kaggle/working/results" if ON_KAGGLE else "./results")
for sub in ("figures", "tables", "checkpoints", "history"):
    (RESULTS_DIR / sub).mkdir(parents=True, exist_ok=True)

SYNSET = {"airplane": "02691156", "car": "02958343", "chair": "03001627"}[CATEGORY]
HF_REPO = "eylulpelinkilic/Colored_Point_Clouds"

print(f"SYSTEM      : {SYSTEM_NAME}")
print(f"ON_KAGGLE   : {ON_KAGGLE} | RESULTS_DIR: {RESULTS_DIR}")
print(f"category    : {CATEGORY} ({SYNSET}) | difficulties {DIFFICULTIES}")
print(f"points      : {NUM_POINTS} | {NUM_TRAIN} train / {NUM_VAL} val | seed {SEED}")
print(f"models      : {TRAIN_MODELS}  (all width-matched to ~{PARAM_TARGET:,} params)")
print(f"sampler     : {SAMPLER}" + (f" {DDIM_STEPS} steps" if SAMPLER == "ddim" else f" {T_STEPS} steps")
      + (f" + RePaint(jump={REPAINT_JUMP}x{REPAINT_NJUMP})" if REPAINT else ""))
print(f"RUN_FULL_EXPERIMENT = {RUN_FULL_EXPERIMENT}"
      + ("" if RUN_FULL_EXPERIMENT else "   <- training is skipped until you flip this"))

## §2 · Environment

In [ ]:
import importlib
for m in ["numpy", "torch", "pandas", "scipy", "matplotlib", "huggingface_hub", "plotly"]:
    print(f"{m:16s} {'present' if importlib.util.find_spec(m) else 'MISSING'}")
print()

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy import stats

print(f"numpy {np.__version__} | torch {torch.__version__} | pandas {pd.__version__}")
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"cuda: {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    print(f" | {torch.cuda.get_device_name(0)} | "
          f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")
else:
    print("  (CPU — fine for the sanity check, far too slow for the full run)")

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
print(f"\nseeded with {SEED} | device {DEV}")

## Shared spec — the control that makes this comparison mean anything

Two systems can only be compared if they are scored on the same objects, in the same frame, by the
same code. That is not achievable by "using the same settings" — it drifts the moment one notebook
is edited. So everything shared is defined **once**, embedded verbatim as a string, and hashed:

* data loading and the partial/`_missing` disambiguation
* the deterministic train/val split
* the **partial-frame normalisation** (never ground-truth statistics — see below)
* all 11 metrics, and `spec_score`, the single scoring entry point
* `EQUALIZE_PRED_N`, the density control

Each notebook prints `SPEC_HASH`. **If two notebooks print different hashes, their numbers must not
be compared** — and the comparison notebook refuses to merge them. `spec_config_fingerprint()` does
the same for the settings that must agree (category, difficulty, point count, split sizes, seed,
thresholds).

The spec is also written to `results/shared_spec.py` so it can be read and diffed directly.

**Why the frame comes from the partial alone.** At test time only the partial exists. Normalising by
ground-truth centroid and radius would hand every system the object's true centre and extent — most
of the completion task — and would flatter whichever system exploits it more. One consequence is
visible downstream: some GT points then lie beyond radius 1, so geometry is never clamped.

In [ ]:
SPEC_SRC = r'''
# ============================================================================
# SHARED SPEC v3 — byte-identical in every notebook of this comparison.
# Data loading, the train/val split, the partial-frame normalisation, and every
# metric live HERE and nowhere else. Each notebook prints SPEC_HASH; if two
# notebooks print different hashes their numbers MUST NOT be compared.
# ============================================================================
SPEC_VERSION = 3

_PLY_NP = {"char": "i1", "uchar": "u1", "short": "i2", "ushort": "u2",
           "int": "i4", "uint": "u4", "float": "f4", "double": "f8",
           "int8": "i1", "uint8": "u1", "int16": "i2", "uint16": "u2",
           "int32": "i4", "uint32": "u4", "float32": "f4", "float64": "f8"}


def read_ply_xyzrgb(path):
    """Binary/ascii PLY -> (xyz float32 (N,3), rgb float32 (N,3) in [0,1]).

    The header is parsed rather than assumed, so an unexpected layout raises instead of
    silently producing garbage. Avoids open3d, which is broken in some local envs.
    """
    with open(path, "rb") as fh:
        if fh.readline().strip() != b"ply":
            raise ValueError(f"{path}: not a PLY")
        fmt, props, n, in_vertex = None, [], None, False
        while True:
            ln = fh.readline()
            if not ln:
                raise ValueError(f"{path}: header never ended")
            t = ln.strip().split()
            if not t:
                continue
            k = t[0].lower()
            if k == b"format":
                fmt = t[1].decode()
            elif k == b"element":
                in_vertex = (t[1].lower() == b"vertex")
                if in_vertex:
                    n = int(t[2])
            elif k == b"property" and in_vertex:
                if t[1].lower() == b"list":
                    raise ValueError(f"{path}: list property in vertex element")
                props.append((t[2].decode(), _PLY_NP[t[1].decode().lower()]))
            elif k == b"end_header":
                break
        names = [p[0] for p in props]
        for need in ("x", "y", "z"):
            if need not in names:
                raise ValueError(f"{path}: no '{need}' property (have {names})")
        if fmt == "binary_little_endian":
            arr = np.frombuffer(fh.read(), dtype=np.dtype([(a, "<" + b) for a, b in props]), count=n)
        elif fmt == "binary_big_endian":
            arr = np.frombuffer(fh.read(), dtype=np.dtype([(a, ">" + b) for a, b in props]), count=n)
        elif fmt == "ascii":
            arr = np.loadtxt(fh, dtype=np.dtype([(a, b) for a, b in props]), max_rows=n)
        else:
            raise ValueError(f"{path}: unknown format {fmt}")
    xyz = np.stack([arr["x"], arr["y"], arr["z"]], -1).astype(np.float32)
    cn = [c for c in ("red", "green", "blue") if c in names]
    if len(cn) == 3:
        rgb = np.stack([arr[c] for c in cn], -1).astype(np.float32)
        if any(dict(props)[c] == "u1" for c in cn):
            rgb /= 255.0
    else:
        rgb = np.full_like(xyz, 0.5)
    return xyz, np.clip(rgb, 0, 1)


def spec_fetch_data():
    """Local tree or HuggingFace snapshot holding labeled_s3/ and occluded_occ/."""
    for cand in [Path("data"), Path("/kaggle/input/colored-point-clouds"),
                 Path(os.environ.get("PCC_DATA_ROOT", "___none___"))]:
        if (cand / "labeled_s3" / SYNSET).is_dir() and (cand / "occluded_occ").is_dir():
            return cand, "local"
    from huggingface_hub import snapshot_download
    pats = [f"labeled_s3/{SYNSET}/*.npz"] + [f"occluded_occ/{d}/{SYNSET}/*.ply" for d in DIFFICULTIES]
    return Path(snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=pats)), "hf"


def spec_model_ids(root):
    """Model ids, sorted, with the partial/_missing trap handled and asserted."""
    all_ply = sorted((root / "occluded_occ" / DIFFICULTIES[0] / SYNSET).glob("*.ply"))
    partial = [p for p in all_ply if not p.stem.endswith("_missing")]
    assert len(partial) * 2 == len(all_ply), (
        f"partial/_missing pairing is not 1:1 ({len(partial)} vs {len(all_ply)}) — "
        "a naive glob('*.ply') returns BOTH files per model")
    return [p.stem for p in partial], len(all_ply)


def spec_visible_mask(gt_xyz, part_xyz, tol=1e-5):
    """Which GT rows appear in the partial. The partial is an exact subset (verified), so
    this mask is exact — it is not a fuzzy nearest-neighbour assignment."""
    d, i = cKDTree(gt_xyz).query(part_xyz, k=1)
    ok = d < tol
    m = np.zeros(len(gt_xyz), bool)
    m[i[ok]] = True
    return m, dict(matched=int(ok.sum()), n_partial=len(part_xyz),
                   max_d=float(d.max()), unique=int(len(np.unique(i[ok]))))


def spec_load_one(root, mid, difficulty):
    z = np.load(root / "labeled_s3" / SYNSET / f"{mid}.npz")
    xyz, rgb = z["xyz"].astype(np.float32), z["rgb"].astype(np.float32)
    part = z["part"].astype(np.int64)
    pxyz, _ = read_ply_xyzrgb(root / "occluded_occ" / difficulty / SYNSET / f"{mid}.ply")
    m, diag = spec_visible_mask(xyz, pxyz)
    return dict(mid=mid, xyz=xyz, rgb=rgb, part=part, vis=m, diag=diag)


def spec_subsample(n_have, n_want, rng):
    if n_have >= n_want:
        return rng.choice(n_have, n_want, replace=False)
    return np.concatenate([np.arange(n_have), rng.choice(n_have, n_want - n_have, replace=True)])


def spec_build_entry(root, mid, difficulty, rng):
    """One completion example in the PARTIAL's frame.

    The frame comes from the partial ALONE. Normalising by ground-truth centroid/radius
    would hand any model the object's true centre and extent — most of the completion task.
    Consequence: GT points can lie beyond radius 1, so geometry is never clamped downstream.
    """
    r = spec_load_one(root, mid, difficulty)
    idx = spec_subsample(len(r["xyz"]), NUM_POINTS, rng)
    xyz, rgb, part, vis = r["xyz"][idx], r["rgb"][idx], r["part"][idx], r["vis"][idx]
    if vis.sum() < 16 or (~vis).sum() < 16:
        return None
    mu = xyz[vis].mean(0)
    rad = max(float(np.linalg.norm(xyz[vis] - mu, axis=1).max()), 1e-6)
    g = (xyz - mu) / rad
    c = 2.0 * rgb - 1.0
    return dict(mid=mid, difficulty=difficulty,
                x0=np.concatenate([g, c], -1).astype(np.float32),
                vis=vis, part=part, mu=mu.astype(np.float32), rad=np.float32(rad),
                sub_idx=idx)


def spec_split(root):
    """The train/val split. Deterministic in SEED and identical across notebooks."""
    ids, n_ply = spec_model_ids(root)
    need = NUM_TRAIN + NUM_VAL
    if len(ids) < need:
        raise RuntimeError(f"only {len(ids)} models available, need {need}")
    ids = ids[:need]
    out = {}
    for tag, sel in (("train", ids[:NUM_TRAIN]), ("val", ids[NUM_TRAIN:])):
        rng = np.random.default_rng(abs(hash((SEED, tag))) % (2 ** 32))
        rows = []
        for mid in sel:
            for d in DIFFICULTIES:
                e = spec_build_entry(root, mid, d, rng)
                if e is not None:
                    rows.append(e)
        out[tag] = rows
    assert not (set(e["mid"] for e in out["train"]) & set(e["mid"] for e in out["val"])), \
        "train/val model overlap"
    return out, ids, n_ply


# ---------------------------------------------------------------- colour science
def srgb_to_lab(rgb):
    rgb = np.clip(np.asarray(rgb, np.float64), 0, 1)
    lin = np.where(rgb > 0.04045, ((rgb + 0.055) / 1.055) ** 2.4, rgb / 12.92)
    M = np.array([[0.4124564, 0.3575761, 0.1804375],
                  [0.2126729, 0.7151522, 0.0721750],
                  [0.0193339, 0.1191920, 0.9503041]])
    xyz = lin @ M.T / np.array([0.95047, 1.0, 1.08883])
    e, k = 216 / 24389, 24389 / 27
    f = np.where(xyz > e, np.cbrt(xyz), (k * xyz + 16) / 116)
    return np.stack([116 * f[..., 1] - 16, 500 * (f[..., 0] - f[..., 1]),
                     200 * (f[..., 1] - f[..., 2])], -1)


def ciede2000(lab1, lab2):
    L1, a1, b1 = lab1[..., 0], lab1[..., 1], lab1[..., 2]
    L2, a2, b2 = lab2[..., 0], lab2[..., 1], lab2[..., 2]
    C1, C2 = np.hypot(a1, b1), np.hypot(a2, b2)
    Cb = (C1 + C2) / 2
    G = 0.5 * (1 - np.sqrt(Cb ** 7 / (Cb ** 7 + 25.0 ** 7 + 1e-30)))
    a1p, a2p = (1 + G) * a1, (1 + G) * a2
    C1p, C2p = np.hypot(a1p, b1), np.hypot(a2p, b2)
    h1p, h2p = np.degrees(np.arctan2(b1, a1p)) % 360, np.degrees(np.arctan2(b2, a2p)) % 360
    dLp, dCp = L2 - L1, C2p - C1p
    dh = h2p - h1p
    dh = np.where(C1p * C2p == 0, 0, np.where(dh > 180, dh - 360, np.where(dh < -180, dh + 360, dh)))
    dHp = 2 * np.sqrt(C1p * C2p) * np.sin(np.radians(dh / 2))
    Lbp, Cbp = (L1 + L2) / 2, (C1p + C2p) / 2
    hs = h1p + h2p
    hbp = np.where(C1p * C2p == 0, hs,
                   np.where(np.abs(h1p - h2p) <= 180, hs / 2,
                            np.where(hs < 360, (hs + 360) / 2, (hs - 360) / 2)))
    T = (1 - 0.17 * np.cos(np.radians(hbp - 30)) + 0.24 * np.cos(np.radians(2 * hbp))
         + 0.32 * np.cos(np.radians(3 * hbp + 6)) - 0.20 * np.cos(np.radians(4 * hbp - 63)))
    dth = 30 * np.exp(-(((hbp - 275) / 25) ** 2))
    Rc = 2 * np.sqrt(Cbp ** 7 / (Cbp ** 7 + 25.0 ** 7 + 1e-30))
    Sl = 1 + 0.015 * (Lbp - 50) ** 2 / np.sqrt(20 + (Lbp - 50) ** 2)
    Sc, Sh = 1 + 0.045 * Cbp, 1 + 0.015 * Cbp * T
    Rt = -np.sin(np.radians(2 * dth)) * Rc
    return np.sqrt((dLp / Sl) ** 2 + (dCp / Sc) ** 2 + (dHp / Sh) ** 2
                   + Rt * (dCp / Sc) * (dHp / Sh))


# ---------------------------------------------------------------- metrics
def spec_fps(xyz, n, seed=0):
    rng = np.random.default_rng(seed)
    N = len(xyz)
    if N <= n:
        return np.arange(N)
    sel = np.empty(n, np.int64); sel[0] = rng.integers(N)
    d = np.linalg.norm(xyz - xyz[sel[0]], axis=1)
    for i in range(1, n):
        sel[i] = int(d.argmax())
        d = np.minimum(d, np.linalg.norm(xyz - xyz[sel[i]], axis=1))
    return sel


def spec_swd(A, B, n_proj=128, seed=0):
    """Sliced Wasserstein-1. No correspondence — a permutation cannot move it."""
    rng = np.random.default_rng(seed)
    P = rng.normal(size=(A.shape[1], n_proj))
    P /= np.linalg.norm(P, axis=0, keepdims=True)
    a, b = np.sort(A @ P, 0), np.sort(B @ P, 0)
    n = min(len(a), len(b))
    qa = a[np.linspace(0, len(a) - 1, n).round().astype(int)]
    qb = b[np.linspace(0, len(b) - 1, n).round().astype(int)]
    return float(np.abs(qa - qb).mean())


def spec_edge_set(xyz, lab, k=8, pct=90):
    n = len(xyz)
    if n < k + 1:
        return np.zeros(n, bool)
    _, idx = cKDTree(xyz).query(xyz, k=min(k + 1, n))
    contrast = ciede2000(lab[idx[:, 1:]], lab[:, None, :]).max(1)
    return contrast >= np.percentile(contrast, pct)


METRIC_DIR = {"cd_l1": "lower", "cd_l2": "lower", "fscore": "higher",
              "dE00_sym": "lower", "dE00_gt2pred": "lower", "dE00_pred2gt": "lower",
              "dE00_matched": "lower", "match_rate": "higher",
              "colour_swd": "lower", "region_dE": "lower", "colour_edge_iou": "higher"}


def completion_metrics(pred, gt, *, tau=FSCORE_TAU, match_tau=MATCH_TAU, seed=0, n_cells=64):
    """pred/gt: (N,6) — geometry in the PARTIAL's frame, colour as 2*rgb-1.

    Three tiers, because completion has no point correspondence:
      1 geometry            cd_l1, cd_l2, fscore
      2 colour VIA a match  dE00_* (+ match_rate) — partly a geometry metric, by construction
      3 colour WITHOUT one  colour_swd, region_dE, colour_edge_iou — geometry cannot move these
    NaN (never 0) where a metric is undefined.
    """
    pg, pc = pred[:, :3].astype(np.float64), (pred[:, 3:] + 1) / 2
    gg, gc = gt[:, :3].astype(np.float64), (gt[:, 3:] + 1) / 2
    out = {}
    if len(pg) < 4 or len(gg) < 4:
        return {k: np.nan for k in METRIC_DIR}
    tp, tg = cKDTree(pg), cKDTree(gg)
    d_g2p, i_g2p = tp.query(gg, k=1)
    d_p2g, i_p2g = tg.query(pg, k=1)

    out["cd_l1"] = float(d_g2p.mean() + d_p2g.mean())
    out["cd_l2"] = float((d_g2p ** 2).mean() + (d_p2g ** 2).mean())
    prec, rec = float((d_p2g < tau).mean()), float((d_g2p < tau).mean())
    out["fscore"] = 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)

    lab_g, lab_p = srgb_to_lab(gc), srgb_to_lab(pc)
    transferred = lab_p[i_g2p]
    dE = ciede2000(transferred, lab_g)
    out["dE00_gt2pred"] = float(dE.mean())
    out["dE00_pred2gt"] = float(ciede2000(lab_p, lab_g[i_p2g]).mean())
    out["dE00_sym"] = 0.5 * (out["dE00_gt2pred"] + out["dE00_pred2gt"])
    ok = d_g2p < match_tau
    out["match_rate"] = float(ok.mean())
    out["dE00_matched"] = float(dE[ok].mean()) if ok.sum() >= 8 else np.nan

    out["colour_swd"] = spec_swd(lab_p, lab_g, seed=seed)
    cells = spec_fps(gg, min(n_cells, len(gg)), seed=seed)
    tc = cKDTree(gg[cells])
    cg, cp = tc.query(gg, k=1)[1], tc.query(pg, k=1)[1]
    mg = np.full((len(cells), 3), np.nan); mp = np.full((len(cells), 3), np.nan)
    for c in range(len(cells)):
        a, b = lab_g[cg == c], lab_p[cp == c]
        if len(a): mg[c] = a.mean(0)
        if len(b): mp[c] = b.mean(0)
    both = ~np.isnan(mg).any(1) & ~np.isnan(mp).any(1)
    out["region_dE"] = float(ciede2000(mp[both], mg[both]).mean()) if both.sum() >= 4 else np.nan

    eg, ep = spec_edge_set(gg, lab_g), spec_edge_set(gg, transferred)
    u = (eg | ep).sum()
    out["colour_edge_iou"] = float((eg & ep).sum() / u) if u else np.nan
    return out


EVAL_COLUMNS = ["system", "method", "rep", "obj", "mid", "difficulty", "region",
                "n_pred", "n_gt"] + list(METRIC_DIR)


def spec_score(pred, gt, *, seed=0, equalize_to=None):
    """Score ONE completion. This is the only scoring entry point either notebook may call.

    `equalize_to` FPS-downsamples the prediction before scoring. It exists because the two
    systems under comparison do not generate the same number of points — Joint6D returns the
    N-point cloud it diffused, while the PoinTr+RePaint pipeline returns a union of the visible
    partial and PoinTr's fixed-size completion. Chamfer and F-score both move with point count,
    so comparing them raw would partly measure density rather than quality. Setting
    EQUALIZE_PRED_N puts both at the same count; n_pred is always recorded either way, so the
    choice is visible in the output rather than buried.
    """
    if equalize_to == "gt":
        equalize_to = len(gt)          # match the GT's own count for this object
    if equalize_to and len(pred) > equalize_to:
        pred = pred[spec_fps(pred[:, :3].astype(np.float64), equalize_to, seed=seed)]
    m = completion_metrics(pred, gt, seed=seed)
    m["n_pred"], m["n_gt"] = len(pred), len(gt)
    return m


def spec_config_fingerprint():
    """The config values that MUST match for two runs to be comparable."""
    keys = dict(spec=SPEC_VERSION, category=CATEGORY, synset=SYNSET,
                difficulties=list(DIFFICULTIES), num_points=NUM_POINTS,
                num_train=NUM_TRAIN, num_val=NUM_VAL, seed=SEED,
                fscore_tau=FSCORE_TAU, match_tau=MATCH_TAU,
                equalize_pred_n=EQUALIZE_PRED_N, eval_repeats=EVAL_REPEATS)
    blob = json.dumps(keys, sort_keys=True)
    return keys, hashlib.sha256(blob.encode()).hexdigest()[:16]

'''


import hashlib, json
SPEC_HASH = hashlib.sha256(SPEC_SRC.encode()).hexdigest()[:16]
exec(compile(SPEC_SRC, "<shared_spec>", "exec"), globals())

(RESULTS_DIR / "shared_spec.py").write_text(SPEC_SRC)
CFG_KEYS, CFG_HASH = spec_config_fingerprint()
print(f"SHARED SPEC v{SPEC_VERSION}")
print(f"  code   hash : {SPEC_HASH}")
print(f"  config hash : {CFG_HASH}")
print(f"  {len(METRIC_DIR)} metrics | scoring entry point: spec_score(...)")
print()
for k, v in CFG_KEYS.items():
    print(f"    {k:16s} {v}")
print("\nBoth hashes must match across the notebooks you intend to compare.")
json.dump(dict(spec_hash=SPEC_HASH, cfg_hash=CFG_HASH, cfg=CFG_KEYS, system=SYSTEM_NAME),
          open(RESULTS_DIR / "run_manifest.json", "w"), indent=2)

## §3 · Data

The spec's loader is used verbatim. Three properties of this data are **verified rather than
assumed**, because each one is a trap:

1. The partial is an **exact subset** of the GT rows, so the visibility mask is exact and the
   completion target is unambiguous.
2. It is **not index-aligned** — `build_occluded.py` sorts by distance to a random viewpoint before
   splitting, so a KD-tree match is required.
3. Partial and missing files share a directory *and* an extension: a naive glob returns **twice**
   the expected count. The spec asserts the 1:1 pairing.

In [ ]:
t0 = time.time()
DATA_ROOT, SOURCE = spec_fetch_data()
SPLITS, IDS, N_PLY = spec_split(DATA_ROOT)
TRAIN, VAL = SPLITS["train"], SPLITS["val"]
print(f"source   : {SOURCE}  ({DATA_ROOT})")
print(f"models   : {len(IDS)} used | {N_PLY} .ply files -> {N_PLY//2} partials after dropping "
      f"'_missing'")
print(f"built in {time.time()-t0:.1f}s  |  {len(TRAIN)} train / {len(VAL)} val examples")

print("\nverifying partial->GT correspondence on 6 models:\n")
print(f"{'model':>22} {'n_gt':>6} {'n_partial':>10} {'matched':>8} {'unique':>7} {'max_d':>9} {'vis%':>6}")
for mid in IDS[:6]:
    r = spec_load_one(DATA_ROOT, mid, DIFFICULTIES[0]); d = r["diag"]
    exact = (d["matched"] == d["n_partial"] == d["unique"])
    assert exact, f"{mid}: partial is not an exact subset — the mask would be wrong"
    print(f"{mid[:22]:>22} {len(r['xyz']):>6} {d['n_partial']:>10} {d['matched']:>8} "
          f"{d['unique']:>7} {d['max_d']:>9.2e} {100*r['vis'].mean():>5.1f}%")
print("\n-> exact, duplicate-free subset for every model checked: PASS")

X0_TRAIN = torch.from_numpy(np.stack([e["x0"] for e in TRAIN]))
VIS_TRAIN = torch.from_numpy(np.stack([e["vis"] for e in TRAIN]))
X0_VAL   = torch.from_numpy(np.stack([e["x0"] for e in VAL]))
VIS_VAL  = torch.from_numpy(np.stack([e["vis"] for e in VAL]))

## §4 · Channel balance

With six channels in one diffusion, whichever has the larger variance dominates the loss. A
per-channel standardisation puts geometry and colour on equal footing. This is **System-A-only** —
it is an internal representation choice, undone before anything is scored, so it cannot affect the
comparison.

In [ ]:
BALANCE_CHANNELS = True
CH_STD = (X0_TRAIN.reshape(-1, 6).std(0).clamp(min=1e-6) if BALANCE_CHANNELS else torch.ones(6))
X0_TRAIN, X0_VAL = X0_TRAIN / CH_STD, X0_VAL / CH_STD

print(f"visible fraction: train {VIS_TRAIN.float().mean():.3f}  val {VIS_VAL.float().mean():.3f}"
      f"   (difficulty {DIFFICULTIES} -> that fraction of points is GIVEN)")
print(f"\nchannel scales (std before balancing):  "
      + "  ".join(f"{n}={v:.3f}" for n, v in zip("xyzrgb", CH_STD.tolist())))
_r = X0_TRAIN[..., :3].mul(CH_STD[:3]).norm(dim=-1)
print(f"\nGT radius in the PARTIAL's frame: median {_r.median():.3f}  "
      f"p99 {_r.flatten().quantile(.99):.3f}  max {_r.max():.3f}")
print(f"fraction of GT points beyond the partial's radius (|r|>1): {(_r > 1).float().mean():.1%}")
print("-> why geometry is never clamped to [-1,1] in the x0 reconstruction.")


def to_raw(x):
    """balanced units -> the spec's scoring units (xyz in the partial frame, colour 2*rgb-1)."""
    return (x.detach().cpu() * CH_STD).numpy()

In [ ]:
# ---------------- figure: the completion task itself ----------------
def _scat(ax, xyz, c, title, s=3):
    o = np.argsort(xyz[:, 1])
    ax.scatter(xyz[o, 0], xyz[o, 2], c=np.clip(c[o], 0, 1), s=s, linewidths=0)
    ax.set_title(title, fontsize=9); ax.set_aspect("equal"); ax.axis("off")


n_show = min(4, len(VAL))
fig, axes = plt.subplots(3, n_show, figsize=(3.1 * n_show, 9))
axes = np.atleast_2d(axes)
for j in range(n_show):
    e = VAL[j]; g = e["x0"][:, :3]; c = (e["x0"][:, 3:] + 1) / 2
    _scat(axes[0, j], g[e["vis"]], c[e["vis"]], f"INPUT · partial ({e['vis'].mean():.0%} seen)")
    _scat(axes[1, j], g[~e["vis"]], c[~e["vis"]], "TARGET · missing region")
    _scat(axes[2, j], g, c, "GT · complete")
for ax, lab in zip(axes[:, 0], ["given", "to generate", "ground truth"]):
    ax.text(-0.08, .5, lab, transform=ax.transAxes, rotation=90, va="center", fontsize=10)
fig.suptitle(f"The completion task — {CATEGORY}, {DIFFICULTIES[0]}", fontsize=11)
fig.tight_layout(); fig.savefig(RESULTS_DIR / "figures/04_task.png", dpi=130, bbox_inches="tight")
plt.close(fig)
print("saved figures/04_task.png")

## §5 · Is the completion task well-posed?

Two checks before any model: that the missing region is a contiguous crop rather than random
dropout, and that the visible half is not colour-biased relative to the whole object.

In [ ]:
_stats = pd.DataFrame([{
    "difficulty": e["difficulty"], "vis_frac": float(e["vis"].mean()),
    "gt_rgb": float((e["x0"][:, 3:] + 1).mean() / 2),
    "vis_rgb": float((e["x0"][e["vis"], 3:] + 1).mean() / 2),
    "parts_seen": int(len(np.unique(e["part"][e["vis"]]))),
    "parts_total": int(len(np.unique(e["part"]))),
} for e in VAL])
print("validation split, per difficulty:\n")
print(_stats.groupby("difficulty").agg(
    n=("vis_frac", "size"), vis_frac=("vis_frac", "mean"), gt_rgb=("gt_rgb", "mean"),
    vis_rgb=("vis_rgb", "mean"), parts_seen=("parts_seen", "mean"),
    parts_total=("parts_total", "mean")).round(3).to_string())
print("\nvis_rgb ~ gt_rgb  => the visible half is not colour-biased relative to the whole object.")
print("parts_seen < parts_total => whole semantic parts are sometimes entirely unobserved, so")
print("their colour must be invented from context. Part labels are NEVER given to a model here;")
print("they are loaded for this diagnostic and the figures only.")

_e = VAL[0]; _g = _e["x0"][:, :3]
_d_mm = cKDTree(_g[~_e["vis"]]).query(_g[~_e["vis"]], k=2)[0][:, 1].mean()
_d_all = cKDTree(_g).query(_g, k=2)[0][:, 1].mean()
print(f"\nmissing-region NN spacing {_d_mm:.4f} vs whole-cloud {_d_all:.4f} "
      f"(ratio {_d_mm/_d_all:.2f}) -> ratio near 1 means a contiguous crop, not random dropout.")

## §6 · Joint 6-D diffusion

One DDPM over all six channels; `q_sample` and the $\hat{x}_0$ reconstruction are shared by every
variant, so no variant can differ through its noising. The reconstruction clamps **colour only** —
geometry is left free for the reason measured in §4.

In [ ]:
def cosine_betas(T, s=0.008):
    t = torch.linspace(0, T, T + 1, dtype=torch.float64) / T
    f = torch.cos((t + s) / (1 + s) * math.pi / 2) ** 2
    return torch.clip(1 - f[1:] / f[:-1], 1e-8, 0.999).float()


def linear_betas(T):
    return torch.linspace(1e-4, 0.02, T)


class Diffusion:
    def __init__(self, T=T_STEPS, schedule=SCHEDULE, device=DEV):
        self.T = T
        self.betas = (cosine_betas(T) if schedule == "cosine" else linear_betas(T)).to(device)
        self.alphas = 1 - self.betas
        self.abar = torch.cumprod(self.alphas, 0)
        self.sqrt_ab, self.sqrt_1mab = self.abar.sqrt(), (1 - self.abar).sqrt()
        ab_prev = torch.cat([torch.ones(1, device=device), self.abar[:-1]])
        self.ab_prev = ab_prev
        self.post_var = self.betas * (1 - ab_prev) / (1 - self.abar)
        self.post_c0 = self.betas * ab_prev.sqrt() / (1 - self.abar)
        self.post_ct = (1 - ab_prev) * self.alphas.sqrt() / (1 - self.abar)

    def q_sample(self, x0, t, noise):
        return self.sqrt_ab[t].view(-1, 1, 1) * x0 + self.sqrt_1mab[t].view(-1, 1, 1) * noise

    def x0_from_eps(self, x_t, t, eps, clamp=True):
        """Reconstruct x0. At high t this divides by sqrt(abar) ~ 1e-4, so an unbounded
        reconstruction DIVERGES during sampling — bounds are required, but they must not be
        so tight that legitimate points are cut off (see §4: ~5% of GT lies beyond radius 1).
        So geometry is clamped to a data-derived envelope, not to the unit sphere, and colour
        to its true per-channel bound."""
        a, s = self.sqrt_ab[t].view(-1, 1, 1), self.sqrt_1mab[t].view(-1, 1, 1)
        x0 = (x_t - s * eps) / a
        if clamp:
            x0 = torch.cat([x0[..., :3].clamp(-GEOM_LIM, GEOM_LIM),
                            x0[..., 3:].clamp(-COL_LIM.to(x0.device), COL_LIM.to(x0.device))], -1)
        return x0


COL_LIM = (1.0 / CH_STD[3:]).clone()      # colour is 2*rgb-1 in [-1,1]; express that in balanced units
# geometry envelope: generous enough that no real point is clipped, finite enough that sampling
# cannot run away. Derived from the training data, then reported so the choice is auditable.
GEOM_MARGIN = 1.25
GEOM_LIM = float(X0_TRAIN[..., :3].abs().max() * GEOM_MARGIN)
DIF = Diffusion()
PROBE_T = sorted({0, *(int(T_STEPS * f) for f in (.05, .1, .25, .5, .75, .9)), T_STEPS - 1})
EVAL_T = [t for t in (int(T_STEPS * f) for f in (.05, .1, .25, .5, .75, .9)) if 0 < t < T_STEPS]

print(f"abar[0]={DIF.abar[0]:.6f}  abar[T/2]={DIF.abar[T_STEPS//2]:.6f}  abar[T-1]={DIF.abar[-1]:.3e}")
print("colour clamp per channel (balanced units): "
      + "  ".join(f"{n}=±{v:.3f}" for n, v in zip("rgb", COL_LIM.tolist())))
_gmax = float(X0_TRAIN[..., :3].abs().max())
print(f"geometry clamp: ±{GEOM_LIM:.3f}  (train max |coord| {_gmax:.3f} x {GEOM_MARGIN} margin)")
print(f"  fraction of TRAIN geometry outside the clamp: "
      f"{(X0_TRAIN[..., :3].abs() > GEOM_LIM).float().mean():.4%}  <- must be 0")
assert (X0_TRAIN[..., :3].abs() > GEOM_LIM).float().mean() == 0, "clamp would cut real points"

_x0 = X0_VAL[:4].to(DEV)
print("\nround-trip  x0 -> q_sample -> x0_from_eps(true eps)   [unclamped]")
for t in PROBE_T:
    tt = torch.full((4,), t, dtype=torch.long, device=DEV)
    n = torch.randn_like(_x0)
    err = (DIF.x0_from_eps(DIF.q_sample(_x0, tt, n), tt, n, clamp=False) - _x0).abs().max()
    print(f"  t={t:4d}  sqrt(abar)={DIF.sqrt_ab[t]:.5f}   max|err| = {err:.2e}")
    assert err < 1e-2, f"forward/reverse inconsistent at t={t}"
print("\nDDPM forward process: all checks passed.")

In [ ]:
_sig = X0_TRAIN.reshape(-1, 6).std(0)
SNR = pd.DataFrame([dict(t=t, sqrt_abar=round(float(DIF.sqrt_ab[t]), 4),
                         sqrt_1mabar=round(float(DIF.sqrt_1mab[t]), 4),
                         snr_xyz=round(float(DIF.sqrt_ab[t] * _sig[:3].mean() / DIF.sqrt_1mab[t]), 4),
                         snr_rgb=round(float(DIF.sqrt_ab[t] * _sig[3:].mean() / DIF.sqrt_1mab[t]), 4))
                    for t in EVAL_T])
print(SNR.to_string(index=False))
print(f"\nchannel balancing = {BALANCE_CHANNELS}: the two SNR columns should be nearly identical,")
print("so neither modality is effectively noisier than the other at any t.")
SNR.to_csv(RESULTS_DIR / "tables/snr_by_timestep.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
tt = np.arange(T_STEPS)
ax[0].plot(tt, DIF.abar.cpu(), lw=2); ax[0].set_title(f"$\\bar\\alpha_t$ ({SCHEDULE})")
ax[0].set_xlabel("t"); ax[0].grid(alpha=.3)
ax[1].semilogy(tt, (DIF.sqrt_ab / DIF.sqrt_1mab).cpu(), lw=2)
ax[1].axhline(1, color="crimson", ls="--", lw=1, label="SNR = 1")
ax[1].set_title("signal-to-noise ratio"); ax[1].set_xlabel("t"); ax[1].legend(); ax[1].grid(alpha=.3)
fig.tight_layout(); fig.savefig(RESULTS_DIR / "figures/06_schedule.png", dpi=130); plt.close(fig)
print("saved figures/06_schedule.png")

## §7 · Metric self-tests

The metrics come from the shared spec; what happens here is **verification**. Both feasibility
studies were burned once by a metric that looked reasonable and measured nothing, so each tier is
checked against inputs whose correct score is known in advance.

In [ ]:
_T1 = np.array([[50, 2.6772, -79.7751], [50, 3.1571, -77.2803], [50, 2.8361, -74.0200],
                [50, -1.3802, -84.2814], [50, 0, 0], [50, -1, 2], [50, 2.5, 0]])
_T2 = np.array([[50, 0, -82.7485], [50, 0, -82.7485], [50, 0, -82.7485],
                [50, 0, -82.7485], [50, -1, 2], [50, 0, 0], [50, 0, -2.5]])
_REF = np.array([2.0425, 2.8615, 3.4412, 1.0000, 2.3669, 2.3669, 4.3065])
print("CIEDE2000 self-test (Sharma et al. 2005 reference pairs):")
_got = ciede2000(_T1, _T2)
for g, r in zip(_got, _REF):
    print(f"  computed {g:8.4f}   reference {r:8.4f}   |diff| {abs(g-r):.2e}")
assert np.allclose(_got, _REF, atol=1e-3), "CIEDE2000 implementation is wrong"
print("-> PASS")

In [ ]:
print("\nmetric behaviour on inputs with a KNOWN correct answer:\n")
print(f"{'':22} {'exact copy':>12} {'colour scrambled':>18} {'geometry jittered':>19}")
rows, rng = [], np.random.default_rng(0)
for e in VAL[:6]:
    x = e["x0"]
    scram = x.copy(); scram[:, 3:] = scram[rng.permutation(len(x)), 3:]
    jit = x.copy(); jit[:, :3] += rng.normal(0, .05, (len(x), 3)).astype(np.float32)
    rows.append((spec_score(x, x), spec_score(scram, x), spec_score(jit, x)))
for k in METRIC_DIR:
    a, b, c = (np.mean([r[i][k] for r in rows]) for i in range(3))
    print(f"  {k:20s} {a:12.5f} {b:18.5f} {c:19.5f}")

_ex = {k: np.mean([r[0][k] for r in rows]) for k in METRIC_DIR}
_sc = {k: np.mean([r[1][k] for r in rows]) for k in METRIC_DIR}
assert _ex["cd_l1"] < 1e-6 and _ex["fscore"] > .999, "geometry metrics fail on an exact copy"
assert _ex["dE00_sym"] < 1e-3, "colour metric fails on an exact copy"
assert _ex["colour_edge_iou"] > .999, "edge IoU fails on an exact copy"
assert _sc["dE00_sym"] > 1.0, "scrambling colour did not move dE00 — the metric is blind"
assert _sc["colour_swd"] < 1e-6, "SWD moved under a permutation — it must be distribution-only"
assert _sc["cd_l1"] < 1e-6, "scrambling colour moved a GEOMETRY metric — tiers are entangled"
print("""
PASS. The three columns are the point:
  exact copy        every metric at its perfect value
  colour scrambled  dE00 and region_dE rise sharply; colour_swd stays ~0 and the geometry
                    metrics stay at 0 — a permutation changes WHERE colours are, not WHICH
                    exist, and touches geometry not at all
  geometry jittered geometry metrics degrade, and the tier-2 colour metrics degrade WITH them,
                    which is precisely why tier 3 exists and why dE00 is never read alone
""")

## §8 · The model ladder

Because the partial is an exact subset of the GT points (§3), completion here is **masked
inpainting**: every model sees the same $N$ points and is told which ones are given. Input is

$$\big[\; \underbrace{x_t}_{6} \;\;\big|\;\; \underbrace{v}_{1} \;\;\big|\;\; \underbrace{v \odot x_0}_{6} \;\big]$$

— the noisy cloud, the visibility indicator, and the **clean known values** zeroed out where nothing
is observed. All three variants receive exactly this; they differ only in **how information is
allowed to move between points**, which is the variable the feasibility studies isolated.

| variant | how information moves between points | the finding it embodies |
|---|---|---|
| `J6D-pw` | **global max-pool only** — every point sees one summary of the whole cloud | the analogue of Study 2's `C+Gpw`: conditioning present, *local* aggregation absent |
| `J6D-self` | + EdgeConv over kNN of the **noisy** cloud | ordinary local aggregation |
| `J6D` | + a second EdgeConv branch over the nearest **observed** points, carrying their **clean** geometry | Question A2: clean geometry aggregated over neighbourhoods, **-63.2 %**, win 40/40 |

**`J6D-pw` is not a no-communication baseline, and must not be described as one.** The global
max-pool is present in *every* variant, ungated — and since the observed values are part of the
input, that pool is a live channel carrying visible colour to missing points. So Q1 measures
**global-only versus global-plus-local** aggregation. That is the honest reading, and it is the same
contrast Study 2 ran (its `C` and `C+Gpw` also pooled globally), which is exactly why it is the right
replication target. It is a *stronger* baseline than "pointwise" would suggest, which makes a
positive Q1 result harder to obtain, not easier.

**The extra branch adds a path; it does not restrict the existing one.** That distinction is the
whole lesson of Questions B and C: adding part identity as a channel was neutral, but *restricting*
aggregation to same-part neighbours cost -6.0 % overall and -4.1 % at boundaries, winning on 1 object
in 40. `J6D` therefore only ever adds context — no mask ever removes a neighbour.

### Closing the capacity confound

Study 2's headline came from a model with **+66.8 % more parameters** than its baseline, which is why
it could not be claimed outright. Here `solve_width` tunes each variant's width until all three land
within `PARAM_TOL` of the same budget, so a difference between them cannot be a difference in size.

In [ ]:
def _gather_nb(h, idx):                       # (B,N,C),(B,N,k) -> (B,N,k,C)
    B, N, C = h.shape
    off = (torch.arange(B, device=h.device) * N).view(B, 1, 1)
    return h.reshape(B * N, C)[(idx + off).reshape(-1)].reshape(B, N, idx.shape[-1], C)


def knn_self(xyz, k, chunk=1024):
    """(B,N,3) -> (B,N,k) neighbour indices within the same cloud, self excluded."""
    B, N, _ = xyz.shape
    out = torch.empty(B, N, k, dtype=torch.long, device=xyz.device)
    kk = min(k + 1, N)
    for s in range(0, N, chunk):
        d = torch.cdist(xyz[:, s:s + chunk], xyz)
        idx = d.topk(kk, dim=-1, largest=False).indices[:, :, 1:]
        if idx.shape[-1] < k:
            idx = idx[..., [i % idx.shape[-1] for i in range(k)]]
        out[:, s:s + chunk] = idx
    return out


def knn_visible(query_xyz, key_xyz, vis, k, chunk=1024):
    """Nearest OBSERVED points. Queries are noisy positions; keys are the clean known ones.

    Points with fewer than k observed neighbours repeat what they have — a query is never
    left without context, and no unobserved point is ever admitted.
    """
    B, N, _ = query_xyz.shape
    out = torch.empty(B, N, k, dtype=torch.long, device=query_xyz.device)
    big = torch.finfo(query_xyz.dtype).max / 4
    for s in range(0, N, chunk):
        d = torch.cdist(query_xyz[:, s:s + chunk], key_xyz)
        d = d.masked_fill(~vis.unsqueeze(1), big)          # only observed points may be chosen
        out[:, s:s + chunk] = d.topk(min(k, N), dim=-1, largest=False).indices
    return out


class Block(nn.Module):
    """Residual block: global context (+ optional self-EdgeConv, + optional observed-EdgeConv),
    FiLM-modulated by the timestep embedding."""

    def __init__(self, w, use_self=False, use_cross=False):
        super().__init__()
        self.use_self, self.use_cross = use_self, use_cross
        if use_self:
            self.edge = nn.Sequential(nn.Linear(2 * w + 4, w), nn.GELU(), nn.Linear(w, w))
        if use_cross:
            # +4 relative geometry, +6 the neighbour's CLEAN 6-D value
            self.cross = nn.Sequential(nn.Linear(2 * w + 4 + 6, w), nn.GELU(), nn.Linear(w, w))
        n_ctx = 2 + int(use_self) + int(use_cross)
        self.fuse = nn.Sequential(nn.LayerNorm(w * n_ctx), nn.Linear(w * n_ctx, w),
                                  nn.GELU(), nn.Linear(w, w))
        self.film = nn.Linear(w, 2 * w)

    def forward(self, h, temb, ctx):
        feats = [h, h.max(1, keepdim=True).values.expand_as(h)]
        if self.use_self:
            hj = _gather_nb(h, ctx["idx_self"]); hi = h.unsqueeze(2).expand_as(hj)
            feats.append(self.edge(torch.cat([hi, hj - hi, ctx["rel_self"]], -1)).max(2).values)
        if self.use_cross:
            hj = _gather_nb(h, ctx["idx_cross"]); hi = h.unsqueeze(2).expand_as(hj)
            feats.append(self.cross(torch.cat([hi, hj - hi, ctx["rel_cross"],
                                               ctx["val_cross"]], -1)).max(2).values)
        d = self.fuse(torch.cat(feats, -1))
        sc, sh = self.film(temb).unsqueeze(1).chunk(2, -1)
        return h + d * (1 + sc) + sh


def timestep_embedding(t, dim):
    half = dim // 2
    f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    a = t.float().view(-1, 1) * f.view(1, -1)
    e = torch.cat([a.sin(), a.cos()], -1)
    return F.pad(e, (0, dim - e.shape[-1])) if e.shape[-1] < dim else e

In [ ]:
class Joint6D(nn.Module):
    """eps-prediction over all six channels. Permutation-equivariant and N-agnostic."""

    def __init__(self, width=192, n_blocks=4, mode="full", k_self=KNN_SELF, k_cross=KNN_CROSS):
        super().__init__()
        assert mode in ("pw", "self", "full")
        self.mode, self.width = mode, width
        self.k_self, self.k_cross = k_self, k_cross
        self.use_self = mode in ("self", "full")
        self.use_cross = (mode == "full")
        self.inp = nn.Linear(6 + 1 + 6, width)               # x_t, visibility, known values
        self.temb = nn.Sequential(nn.Linear(width, width), nn.SiLU(), nn.Linear(width, width))
        self.blocks = nn.ModuleList([Block(width, self.use_self, self.use_cross)
                                     for _ in range(n_blocks)])
        self.out = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, width), nn.GELU(),
                                 nn.Linear(width, 6))

    def build_ctx(self, x_t, x_known, vis):
        """Graphs are rebuilt from the CURRENT noisy geometry every call — geometry is being
        generated here, unlike the feasibility studies where it was clean and cached once."""
        ctx = {}
        g_t = x_t[..., :3]
        if self.use_self:
            idx = knn_self(g_t, self.k_self)
            rel = _gather_nb(g_t, idx) - g_t.unsqueeze(2)
            sc = rel.norm(dim=-1).mean((1, 2), keepdim=True).clamp(min=1e-6).unsqueeze(-1)
            ctx["idx_self"] = idx
            ctx["rel_self"] = torch.cat([rel / sc, rel.norm(dim=-1, keepdim=True) / sc], -1)
        if self.use_cross:
            g_k = x_known[..., :3]
            idx = knn_visible(g_t, g_k, vis, self.k_cross)
            rel = _gather_nb(g_k, idx) - g_t.unsqueeze(2)     # to the CLEAN observed position
            sc = rel.norm(dim=-1).mean((1, 2), keepdim=True).clamp(min=1e-6).unsqueeze(-1)
            ctx["idx_cross"] = idx
            ctx["rel_cross"] = torch.cat([rel / sc, rel.norm(dim=-1, keepdim=True) / sc], -1)
            ctx["val_cross"] = _gather_nb(x_known, idx)       # the neighbour's clean 6-D value
        return ctx

    def forward(self, x_t, t, x_known, vis, ctx=None):
        if ctx is None:
            ctx = self.build_ctx(x_t, x_known, vis)
        v = vis.unsqueeze(-1).float()
        h = self.inp(torch.cat([x_t, v, x_known * v], -1))
        temb = self.temb(timestep_embedding(t.expand(x_t.shape[0]) if t.dim() == 0 else t, self.width))
        for b in self.blocks:
            h = b(h, temb, ctx)
        return self.out(h)


MODEL_MODE = {"J6D-pw": "pw", "J6D-self": "self", "J6D": "full"}


def n_params(m):
    return sum(p.numel() for p in m.parameters())


def solve_width(mode, target=PARAM_TARGET, lo=32, hi=768):
    """Smallest width whose parameter count is closest to `target`. Closes the A2 capacity confound."""
    best, bw = None, lo
    while lo <= hi:
        mid = (lo + hi) // 2
        n = n_params(Joint6D(width=mid, n_blocks=N_BLOCKS, mode=mode))
        if best is None or abs(n - target) < abs(best - target):
            best, bw = n, mid
        if n < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return bw, best


WIDTHS = {}
print(f"solving widths for a {PARAM_TARGET:,}-parameter budget:\n")
print(f"{'model':10} {'mode':6} {'width':>6} {'params':>12} {'off target':>11}")
for k in TRAIN_MODELS:
    w, n = solve_width(MODEL_MODE[k])
    WIDTHS[k] = w
    off = (n - PARAM_TARGET) / PARAM_TARGET
    print(f"{k:10} {MODEL_MODE[k]:6} {w:>6} {n:>12,} {off:>10.2%}")
    assert abs(off) <= PARAM_TOL, f"{k} is {off:.1%} off the budget — widen the search or loosen PARAM_TOL"


def build_model(key):
    return Joint6D(width=WIDTHS[key], n_blocks=N_BLOCKS, mode=MODEL_MODE[key])


_spread = [n_params(build_model(k)) for k in TRAIN_MODELS]
print(f"\nspread across the ladder: {(max(_spread)-min(_spread))/min(_spread):.2%}"
      "  -> differences between these models cannot be capacity differences.")
pd.DataFrame([{"model": k, "mode": MODEL_MODE[k], "width": WIDTHS[k],
               "params": n_params(build_model(k))} for k in TRAIN_MODELS]
             ).to_csv(RESULTS_DIR / "tables/model_parameters.csv", index=False)

### §9 · Wiring tests

Three assertions, run before any training. Each one has a specific failure it is designed to catch,
and the notebook refuses to continue if any fails.

In [ ]:
@torch.no_grad()
def wiring_report():
    B = 2
    x0 = X0_VAL[:B].to(DEV); vis = VIS_VAL[:B].to(DEV)
    t = torch.full((B,), EVAL_T[len(EVAL_T) // 2], dtype=torch.long, device=DEV)
    x_t = DIF.q_sample(x0, t, torch.randn_like(x0))
    xk = x0 * vis.unsqueeze(-1)
    rows = []
    for k in TRAIN_MODELS:
        m = build_model(k).to(DEV).eval()
        base = m(x_t, t, xk, vis)
        assert base.shape == (B, NUM_POINTS, 6), f"{k}: wrong output shape {tuple(base.shape)}"

        # (1) does it use the observed values at all?
        d_known = (m(x_t, t, torch.randn_like(xk) * vis.unsqueeze(-1), vis) - base).abs().mean()
        # (2) is it blind to what is hidden BEYOND the mask? perturb the INVISIBLE slots of x_known,
        #     which a correct model never reads -> must be exactly 0.0
        leak = xk.clone()
        leak[~vis] = torch.randn_like(leak[~vis])
        d_leak = (m(x_t, t, leak, vis) - base).abs().mean()
        # (3) permutation equivariance
        p = torch.randperm(NUM_POINTS, device=DEV)
        d_perm = (m(x_t[:, p], t, xk[:, p], vis[:, p]) - base[:, p]).abs().max()
        rows.append(dict(model=k, uses_known=float(d_known), leak_hidden=float(d_leak),
                         perm_err=float(d_perm)))
    return pd.DataFrame(rows)


W = wiring_report()
W["verdict"] = np.where((W.uses_known > 1e-4) & (W.leak_hidden == 0) & (W.perm_err < 1e-3),
                        "PASS", "FAIL")
print("wiring tests:\n")
print(W.to_string(index=False))
print("""
  uses_known   > 0     the model actually reads the observed values
  leak_hidden == 0.0   perturbing the MASKED-OUT slots changes nothing. This is the one that
                       matters: a non-zero value means ground truth is leaking in through the
                       conditioning and every downstream number would be inflated.
  perm_err    ~= 0     permutation equivariance — no dependence on point order
""")
assert (W.verdict == "PASS").all(), "wiring is broken; fix before training"
W.to_csv(RESULTS_DIR / "tables/wiring_test.csv", index=False)

## §10 · Training

Identical protocol for every variant, so a difference in score is a difference in architecture:

* **paired batches** — one generator draws the shape order, the timesteps and the noise; all models
  replay the same draws in the same order,
* **one frozen validation bank** of `(shape, t, noise)` triples, reused at every evaluation,
* **EMA weights** for sampling (generation is far more sensitive to weight noise than the denoising
  loss is),
* per-epoch checkpoints that resume and are rejected if §1 changed.

`VAL_EVERY` samples the validation curve rather than computing it every epoch — the last epoch is
always evaluated, so the reported final number is exact.

In [ ]:
class EMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}

    def update(self, model):
        if self.decay <= 0:
            return
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)

    def copy_to(self, model):
        if self.decay <= 0:
            return
        sd = model.state_dict()
        for k, v in self.shadow.items():
            sd[k].copy_(v)


# ---------------- frozen validation bank ----------------
VAL_BANK_REPEATS = 4
_g = torch.Generator().manual_seed(SEED + 991)
_n_val = len(X0_VAL)
VAL_BANK = [dict(
    i=int(torch.randint(_n_val, (1,), generator=_g)),
    t=int(torch.randint(1, T_STEPS - 1, (1,), generator=_g)),
    noise=torch.randn(NUM_POINTS, 6, generator=_g),
) for _ in range(_n_val * VAL_BANK_REPEATS)]
print(f"frozen val bank: {len(VAL_BANK)} (shape, t, noise) triples, "
      f"t spans {min(b['t'] for b in VAL_BANK)}..{max(b['t'] for b in VAL_BANK)}")

_VB_X0  = torch.stack([X0_VAL[b["i"]] for b in VAL_BANK])
_VB_VIS = torch.stack([VIS_VAL[b["i"]] for b in VAL_BANK])
_VB_T   = torch.tensor([b["t"] for b in VAL_BANK], dtype=torch.long)
_VB_N   = torch.stack([b["noise"] for b in VAL_BANK])


@torch.no_grad()
def val_loss(model, chunk=8):
    model.eval(); tot = n = 0
    for s in range(0, len(_VB_X0), chunk):
        x0 = _VB_X0[s:s+chunk].to(DEV); vis = _VB_VIS[s:s+chunk].to(DEV)
        t = _VB_T[s:s+chunk].to(DEV); eps = _VB_N[s:s+chunk].to(DEV)
        x_t = DIF.q_sample(x0, t, eps)
        pred = model(x_t, t, x0 * vis.unsqueeze(-1), vis)
        tot += F.mse_loss(pred, eps, reduction="sum").item(); n += eps.numel()
    return tot / n


def train_model(key, epochs=NUM_EPOCHS, resume=True, verbose_every=50, log=True):
    torch.manual_seed(SEED); np.random.seed(SEED)
    model = build_model(key).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=LR * 0.01)
    ema = EMA(model)
    hist, start = [], 0

    cfg = dict(cat=CATEGORY, n=NUM_POINTS, w=WIDTHS[key], nb=N_BLOCKS, ep=epochs,
               tr=len(X0_TRAIN), lr=LR, seed=SEED, diff=DIFFICULTIES)
    ck = RESULTS_DIR / f"checkpoints/{key.replace('+','_').replace('-','_')}.pt"
    if resume and ck.exists():
        s = torch.load(ck, map_location=DEV, weights_only=False)
        if s.get("cfg") == cfg:
            model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"])
            sched.load_state_dict(s["sched"]); ema.shadow = {k: v.to(DEV) for k, v in s["ema"].items()}
            hist, start = s["hist"], s["epoch"] + 1
            if log: print(f"  [{key}] resumed at epoch {start}")
        elif log:
            print(f"  [{key}] checkpoint config differs -> retraining from scratch")

    gen = torch.Generator(device="cpu").manual_seed(SEED + 4242)
    for _ in range(start * max(1, len(X0_TRAIN) // BATCH_SIZE)):
        torch.randint(len(X0_TRAIN), (BATCH_SIZE,), generator=gen)   # keep the paired stream aligned

    t_start = time.time()
    for ep in range(start, epochs):
        model.train(); tot = cnt = 0
        for _ in range(max(1, len(X0_TRAIN) // BATCH_SIZE)):
            bi = torch.randint(len(X0_TRAIN), (BATCH_SIZE,), generator=gen)
            x0 = X0_TRAIN[bi].to(DEV); vis = VIS_TRAIN[bi].to(DEV)
            t = torch.randint(0, T_STEPS, (BATCH_SIZE,), device=DEV)
            eps = torch.randn_like(x0)
            x_t = DIF.q_sample(x0, t, eps)
            loss = F.mse_loss(model(x_t, t, x0 * vis.unsqueeze(-1), vis), eps)
            opt.zero_grad(set_to_none=True); loss.backward()
            if GRAD_CLIP:
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step(); ema.update(model)
            tot += loss.item(); cnt += 1
        sched.step()
        do_val = (ep % VAL_EVERY == 0) or (ep == epochs - 1)
        vl = val_loss(model) if do_val else float("nan")
        hist.append(dict(epoch=ep, train_loss=tot / cnt, val_loss=vl,
                         lr=opt.param_groups[0]["lr"], seconds=time.time() - t_start))
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        ema=ema.shadow, hist=hist, epoch=ep, cfg=cfg), ck)
        if log and (ep % verbose_every == 0 or ep == epochs - 1):
            vs = f"{vl:.4f}" if vl == vl else "  --  "
            print(f"  [{key}] ep {ep:4d}/{epochs}  train {tot/cnt:.4f}  val {vs}", flush=True)
    ema.copy_to(model)
    return model, pd.DataFrame(hist)

### §11 · Sanity check — can each variant learn at all?

A tiny memorisation test on 4 clouds. It is a *wiring* check, not a quality signal: the criterion is
that the loss falls and beats the trivial $\hat\epsilon=0$ predictor, which scores exactly 1.0 at
every $t$. `eps`-MSE **falls with $t$ by construction**, so the recovery of $x_0$ is reported
alongside — the lesson from the first study, where a rising-with-$t$ metric was read backwards.

In [ ]:
def tiny_overfit(key, n_shapes=4, steps=SANITY_STEPS):
    torch.manual_seed(SEED)
    m = build_model(key).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=2e-3)
    x0 = X0_TRAIN[:n_shapes].to(DEV); vis = VIS_TRAIN[:n_shapes].to(DEV)
    xk = x0 * vis.unsqueeze(-1)
    g = torch.Generator(device=DEV).manual_seed(SEED)
    losses = []
    for _ in range(steps):
        t = torch.randint(0, T_STEPS, (n_shapes,), device=DEV, generator=g)
        eps = torch.randn(x0.shape, device=DEV, generator=g)
        loss = F.mse_loss(m(DIF.q_sample(x0, t, eps), t, xk, vis), eps)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        losses.append(loss.item())
    m.eval(); probe = {}
    with torch.no_grad():
        for t_ in (EVAL_T[0], EVAL_T[len(EVAL_T)//2], EVAL_T[-1]):
            t = torch.full((n_shapes,), t_, dtype=torch.long, device=DEV)
            eps = torch.randn(x0.shape, device=DEV, generator=g)
            x_t = DIF.q_sample(x0, t, eps)
            pe = m(x_t, t, xk, vis)
            x0h = DIF.x0_from_eps(x_t, t, pe)
            probe[f"eps_t{t_}"] = F.mse_loss(pe, eps).item()
            # how much of the x0 error is removed vs. just guessing x_t/sqrt(abar)?
            naive = (DIF.x0_from_eps(x_t, t, torch.zeros_like(eps)) - x0).pow(2).mean()
            probe[f"x0_gain_t{t_}"] = 1 - (x0h - x0).pow(2).mean().item() / naive.item()
    return dict(losses=np.array(losses), **probe)


if RUN_SANITY_CHECK:
    print(f"tiny-set overfit — {SANITY_STEPS} steps on 4 clouds\n")
    SANITY, SANITY_SECONDS = {}, {}
    for k in TRAIN_MODELS:
        t0 = time.time(); SANITY[k] = tiny_overfit(k); SANITY_SECONDS[k] = time.time() - t0
        L = SANITY[k]["losses"]; o, e = max(20, len(L)//10), max(20, len(L)//10)
        print(f"  {k:9s} {SANITY_SECONDS[k]:6.1f}s   loss {L[:o].mean():.4f} -> {L[-e:].mean():.4f}",
              flush=True)

    rows = []
    for k in TRAIN_MODELS:
        s = SANITY[k]; L = s["losses"]; o, e = max(20, len(L)//10), max(20, len(L)//10)
        r = dict(model=k, loss_start=L[:o].mean(), loss_end=L[-e:].mean(),
                 **{q: s[q] for q in s if q != "losses"})
        r["drop_ok"] = r["loss_end"] < 0.6 * r["loss_start"]
        r["beats_trivial_ok"] = r["loss_end"] < 1.0
        r["x0_gain_ok"] = s[f"x0_gain_t{EVAL_T[len(EVAL_T)//2]}"] > 0.2
        r["passed"] = bool(r["drop_ok"] and r["beats_trivial_ok"] and r["x0_gain_ok"])
        rows.append(r)
    SAN = pd.DataFrame(rows)
    print("\n" + SAN.round(4).to_string(index=False))
    SAN.to_csv(RESULTS_DIR / "tables/sanity_check.csv", index=False)

    fig, ax = plt.subplots(figsize=(7, 3.2))
    for k in TRAIN_MODELS:
        L = SANITY[k]["losses"]; w = max(1, len(L)//50)
        ax.plot(np.convolve(L, np.ones(w)/w, "valid"), lw=1.5, label=k)
    ax.axhline(1.0, color="crimson", ls="--", lw=1, label=r"trivial $\hat\epsilon=0$")
    ax.set_xlabel("step"); ax.set_ylabel("eps-MSE"); ax.set_yscale("log")
    ax.legend(fontsize=8); ax.grid(alpha=.3); ax.set_title("tiny-set overfit")
    fig.tight_layout(); fig.savefig(RESULTS_DIR / "figures/11_sanity.png", dpi=130); plt.close(fig)

    ok = bool(SAN.passed.all())
    print("\n" + "="*74)
    print("SANITY CHECK PASSED" if ok else "SANITY CHECK FAILED — do not train")
    if ok:
        print("All variants drive the joint 6-D denoising loss down and beat the trivial predictor.")
        print("-> set RUN_FULL_EXPERIMENT = True in §1, re-run §1, then continue.")
    print("="*74)
else:
    SANITY = SANITY_SECONDS = None
    print("sanity check skipped (RUN_SANITY_CHECK = False)")

In [ ]:
# ---------------- runtime estimate from THIS machine's sanity timings ----------------
if RUN_SANITY_CHECK and SANITY_SECONDS:
    steps_ep = max(1, len(X0_TRAIN) // BATCH_SIZE)
    est = {}
    for k in TRAIN_MODELS:
        per_step = SANITY_SECONDS[k] / SANITY_STEPS * (BATCH_SIZE / 4)
        vcost = per_step * (len(VAL_BANK) / BATCH_SIZE) / 3.0 / max(VAL_EVERY, 1)
        est[k] = (per_step * steps_ep + vcost) * NUM_EPOCHS / 60
    # count the ACTUAL network evaluations per sample, jump schedule included — guessing a
    # multiplier here is how sampling budgets end up 2-4x wrong
    _base = (np.linspace(T_STEPS - 1, 0, DDIM_STEPS).round().astype(int).tolist()
             if SAMPLER == "ddim" else list(range(T_STEPS - 1, -1, -1)))
    _visit = []
    for _i, _t in enumerate(_base):
        _visit.append(_t)
        if REPAINT and REPAINT_JUMP and _i and _i % REPAINT_JUMP == 0 and _i + 1 < len(_base):
            for _ in range(REPAINT_NJUMP):
                _b = _base[max(0, _i - REPAINT_JUMP):_i + 1]
                _visit.extend(_b[::-1][1:]); _visit.extend(_b[1:])
    n_steps = len(_visit)
    samp = sum(SANITY_SECONDS[k] / SANITY_STEPS * (len(X0_VAL) / 4) * n_steps
               * EVAL_REPEATS / 3.0 / 60 for k in TRAIN_MODELS)
    print(f"estimated cost at {NUM_EPOCHS} epochs, VAL_EVERY={VAL_EVERY} "
          f"(extrapolated from this session's §11 timings):\n")
    for k in TRAIN_MODELS:
        print(f"  {k:10s} train ~{est[k]:5.1f} min")
    print(f"  {'':10s} ------------------")
    print(f"  {'training':10s}       ~{sum(est.values()):5.1f} min")
    print(f"  {'sampling':10s}       ~{samp:5.1f} min   ({EVAL_REPEATS} repeats x {len(X0_VAL)} objects "
          f"x {n_steps} net evals x {len(TRAIN_MODELS)} models)")
    if REPAINT:
        print(f"  {'':10s}       note: {len(_base)} scheduled steps become {n_steps} network "
              f"evaluations ({n_steps/len(_base):.1f}x) once the RePaint jumps are counted.")
    print(f"  {'TOTAL':10s}       ~{sum(est.values())+samp+8:5.1f} min   (+ ~8 min figures/report)")
    print("\nlevers, cheapest first: SAMPLER='ddim' with fewer DDIM_STEPS; EVAL_REPEATS; VAL_EVERY;")
    print("NUM_EPOCHS; then NUM_POINTS. Drop models from TRAIN_MODELS only as a last resort —")
    print("J6D-pw is the control that makes the J6D number mean something.")

## §12 · Sampling

Reverse diffusion from pure noise, with the observed points **re-injected at their correct noise
level** at every step (RePaint, Lugmayr et al. 2022):

$$x_{t-1} \;=\; v \odot q(x_0^{\text{known}}, t{-}1) \;+\; (1-v) \odot p_\theta(x_t)$$

This is exact here rather than approximate, because the partial is a true subset of the target
points (§3) — the model is never asked to reproduce what it was given, only to fill what it was not.
The optional jump schedule re-noises and re-denoises so the generated region has a chance to
harmonise with the injected one.

DDIM is the default for the evaluation sweep; the full DDPM chain is available and is what the
figures use.

In [ ]:
@torch.no_grad()
def sample(model, x0, vis, *, sampler=SAMPLER, steps=DDIM_STEPS, repaint=REPAINT,
           jump=REPAINT_JUMP, n_jump=REPAINT_NJUMP, seed=0, keep_trace=False, n_trace=8):
    """Complete a batch of partial clouds.

    x0 is read ONLY through `vis` (asserted leak-free in §9). Observed points are re-injected at
    every step and returned verbatim, so all reported error is error on invented points.

    One unified update covers both samplers, which keeps the jump schedule correct for arbitrary
    step spacing:
        x_s = sqrt(ab_s) x0hat + sqrt(1 - ab_s - sigma^2) eps + sigma z
        sigma = eta sqrt((1-ab_s)/(1-ab_t)) sqrt(1 - ab_t/ab_s)
    eta = 0 is deterministic DDIM; eta = 1 recovers the DDPM posterior.
    """
    model.eval()
    B = x0.shape[0]
    g = torch.Generator(device=DEV).manual_seed(seed)
    xk, v3 = x0 * vis.unsqueeze(-1), vis.unsqueeze(-1).float()
    eta = 0.0 if sampler == "ddim" else 1.0

    # descending visit order, then RePaint jumps woven in
    base = (np.linspace(T_STEPS - 1, 0, steps).round().astype(int).tolist()
            if sampler == "ddim" else list(range(T_STEPS - 1, -1, -1)))
    visit = []
    for i, t_ in enumerate(base):
        visit.append(t_)
        if repaint and jump and i and i % jump == 0 and i + 1 < len(base):
            for _ in range(n_jump):
                back = base[max(0, i - jump):i + 1]
                visit.extend(back[::-1][1:])      # climb back up
                visit.extend(back[1:])            # and descend again
    visit.append(-1)                              # sentinel: final x0

    def _ab(i):
        return DIF.abar[i] if i >= 0 else torch.tensor(1.0, device=DEV)

    x = torch.randn(x0.shape, device=DEV, generator=g)
    trace, every = [], max(1, len(visit) // max(n_trace, 1))

    for step, (t_, s_) in enumerate(zip(visit[:-1], visit[1:])):
        t = torch.full((B,), t_, dtype=torch.long, device=DEV)
        if repaint:                                    # known points, noised to the CURRENT level
            x = v3 * DIF.q_sample(xk, t, torch.randn(x.shape, device=DEV, generator=g)) + (1 - v3) * x
        if s_ > t_:                                    # jumped back up -> re-noise forward
            ab_t, ab_s = _ab(t_), _ab(s_)
            x = (ab_s / ab_t).sqrt() * x + (1 - ab_s / ab_t).sqrt() * torch.randn(
                x.shape, device=DEV, generator=g)
            continue
        eps = model(x, t, xk, vis)
        x0h = DIF.x0_from_eps(x, t, eps)
        if keep_trace and step % every == 0:
            trace.append((t_, (v3 * xk + (1 - v3) * x0h).clone().cpu()))
        ab_t, ab_s = _ab(t_), _ab(s_)
        if s_ < 0:
            x = x0h
        else:
            sig = eta * ((1 - ab_s) / (1 - ab_t)).sqrt() * (1 - ab_t / ab_s).clamp(min=0).sqrt()
            x = (ab_s.sqrt() * x0h
                 + (1 - ab_s - sig ** 2).clamp(min=0).sqrt() * eps
                 + sig * torch.randn(x.shape, device=DEV, generator=g))

    x = v3 * xk + (1 - v3) * x
    if keep_trace:
        trace.append((0, x.clone().cpu()))
    return (x, trace) if keep_trace else x


def to_raw(x):
    """balanced units -> (xyz in the partial frame, colour as 2*rgb-1)."""
    return (x.detach().cpu() * CH_STD).numpy()


print("sampler ready.")
print(f"  {SAMPLER.upper()}"
      + (f", {DDIM_STEPS} steps" if SAMPLER == "ddim" else f", {T_STEPS} steps")
      + (f", RePaint jump {REPAINT_JUMP}x{REPAINT_NJUMP}" if REPAINT else ", no RePaint"))
print("  observed points are re-injected every step and returned verbatim, so all reported error")
print("  is error on points the model actually had to invent.")

## §13 · Train

In [ ]:
MODELS, HISTORY = {}, {}
if RUN_FULL_EXPERIMENT:
    for k in TRAIN_MODELS:
        t0 = time.time()
        MODELS[k], HISTORY[k] = train_model(k)
        HISTORY[k].to_csv(RESULTS_DIR / f"history/history_{k.replace('+','_').replace('-','_')}.csv",
                          index=False)
        print(f"{k} trained in {(time.time()-t0)/60:.1f} min\n")
    MODELS_READY = True
    pd.concat([h.assign(model=k) for k, h in HISTORY.items()]).to_csv(
        RESULTS_DIR / "history/history_all.csv", index=False)
else:
    MODELS_READY = False
    print("RUN_FULL_EXPERIMENT is False -> skipping training.")
    print("Run §11 first; if it prints SANITY CHECK PASSED, set RUN_FULL_EXPERIMENT = True in §1,")
    print("re-run §1, then come back.")
print("MODELS_READY =", MODELS_READY)

In [ ]:
if MODELS_READY:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    for k in TRAIN_MODELS:
        h = HISTORY[k]; hv = h.dropna(subset=["val_loss"])
        ax[0].plot(h.epoch, h.train_loss, lw=.9, alpha=.45)
        ax[0].plot(hv.epoch, hv.val_loss, lw=1.8, marker="o", ms=2.5, label=f"{k}")
        ax[1].plot(hv.epoch, hv.val_loss, lw=1.8, label=k)
    ax[0].set_title("training (thin) and validation (thick) loss"); ax[0].set_xlabel("epoch")
    ax[0].set_ylabel("eps-MSE"); ax[0].set_yscale("log"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    ax[1].set_title("validation loss (frozen bank)"); ax[1].set_xlabel("epoch")
    ax[1].set_yscale("log"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(RESULTS_DIR / "figures/13_training.png", dpi=130); plt.close(fig)

    print("final losses (train / val):")
    for k in TRAIN_MODELS:
        h = HISTORY[k]
        print(f"  {k:10s} train {h.train_loss.iloc[-1]:.4f}   "
              f"val {h.dropna(subset=['val_loss']).val_loss.iloc[-1]:.4f}")
    print("\nA variant with LOWER train but HIGHER val than another is exploiting capacity, not")
    print("structure — but note all three have the same parameter count by construction (§8).")

## §14 · Evaluation

Each validation object is completed `EVAL_REPEATS` times with different sampling noise, because
generation is stochastic and a single draw is not an estimate. Every model sees the **same** seeds,
so the comparison stays paired.

Scores are computed on the **missing region** — the points the model had to invent.

In [ ]:
def evaluate_all():
    """Scoring goes through spec_score ONLY, and emits the shared EVAL_COLUMNS schema, so the
    rows can be concatenated with System B's without any reconciliation step."""
    rows, samples = [], {}
    x0_all, vis_all = X0_VAL.to(DEV), VIS_VAL.to(DEV)
    for k in TRAIN_MODELS:
        m = MODELS[k]
        for rep in range(EVAL_REPEATS):
            t0 = time.time()
            preds = []
            for s in range(0, len(x0_all), 8):
                preds.append(sample(m, x0_all[s:s+8], vis_all[s:s+8], seed=SEED + 1000 * rep).cpu())
            pred = torch.cat(preds)
            if rep == 0:
                samples[k] = pred.clone()
            for i in range(len(pred)):
                miss = ~VIS_VAL[i].numpy()
                p_raw, g_raw = to_raw(pred[i]), to_raw(X0_VAL[i])
                meta = dict(system=SYSTEM_NAME, method=k, rep=rep, obj=i, mid=VAL[i]["mid"],
                            difficulty=VAL[i]["difficulty"])
                # the invented points, scored against the region the model had to invent
                rows.append(dict(**meta, region="missing",
                                 **spec_score(p_raw[miss], g_raw[miss], seed=i,
                                              equalize_to=EQUALIZE_PRED_N)))
                rows.append(dict(**meta, region="all",
                                 **spec_score(p_raw, g_raw, seed=i,
                                              equalize_to=EQUALIZE_PRED_N)))
            print(f"  {k:10s} rep {rep+1}/{EVAL_REPEATS} done ({time.time()-t0:.1f}s)", flush=True)
    df = pd.DataFrame(rows)
    return df[[c for c in EVAL_COLUMNS if c in df.columns]], samples


if MODELS_READY:
    t0 = time.time()
    EVAL, SAMPLES = evaluate_all()
    EVAL.to_csv(RESULTS_DIR / "tables/eval_raw.csv", index=False)
    print(f"\n{len(EVAL)} rows in {(time.time()-t0)/60:.1f} min")
    MISS = EVAL[EVAL.region == "missing"]
    MAIN = MISS.groupby("method")[list(METRIC_DIR)].mean().reindex(TRAIN_MODELS)
    print("\n--- MAIN TABLE · missing region only ---\n")
    print(MAIN.round(4).to_string())
    MAIN.to_csv(RESULTS_DIR / "tables/main_table.csv")
else:
    EVAL = MISS = MAIN = SAMPLES = None
    print("skipped — models not trained")

In [ ]:
# ---------------- paired statistics ----------------
def paired(base, cond, df, n_boot=N_BOOT):
    out = []
    for met, direction in METRIC_DIR.items():
        a = df[df.method == base].groupby("obj")[met].mean()
        b = df[df.method == cond].groupby("obj")[met].mean()
        j = pd.concat([a.rename("a"), b.rename("b")], axis=1).dropna()
        if len(j) < 5:
            continue
        sign = -1 if direction == "lower" else 1
        # a metric whose baseline is 0 (e.g. an IoU that never fires) has no meaningful
        # PERCENTAGE change — fall back to the absolute difference and say so in the column.
        denom = abs(j.a.mean())
        rel = denom > 1e-12
        scale = (100.0 / denom) if rel else 1.0
        imp = sign * (j.b - j.a) * scale
        rng = np.random.default_rng(0)
        idx = rng.integers(0, len(j), (n_boot, len(j)))
        ba, bb = j.a.values[idx].mean(1), j.b.values[idx].mean(1)
        with np.errstate(divide="ignore", invalid="ignore"):
            boot = sign * (bb - ba) * (100.0 / np.abs(ba) if rel else 1.0)
        boot = boot[np.isfinite(boot)]
        if boot.size < 20:
            continue
        try:
            p = stats.wilcoxon(j.a, j.b).pvalue
        except Exception:
            p = np.nan
        lo, hi = np.percentile(boot, [2.5, 97.5])
        out.append(dict(metric=met, direction=direction, scale="%" if rel else "abs",
                        baseline=j.a.mean(), conditioned=j.b.mean(),
                        improvement_pct=sign * (j.b.mean() - j.a.mean()) * scale,
                        ci_lo=lo, ci_hi=hi, win_rate=float((imp > 0).mean()), wilcoxon_p=p,
                        verdict="helps" if lo > 0 else ("hurts" if hi < 0 else "unresolved")))
    return pd.DataFrame(out)


COMPARISONS = [
    ("J6D-pw", "J6D-self", "Q1", "Does local aggregation help at all? (the A2 test, on real completion)"),
    ("J6D-self", "J6D", "Q2", "Does a dedicated path to the CLEAN observed geometry add more?"),
    ("J6D-pw", "J6D", "Q3", "Full ladder: pointwise conditioning vs both local paths"),
]

if MODELS_READY:
    STATS = []
    for base, cond, tag, q in COMPARISONS:
        if base not in TRAIN_MODELS or cond not in TRAIN_MODELS:
            continue
        s = paired(base, cond, MISS).assign(comparison=tag, baseline_model=base, cond_model=cond)
        STATS.append(s)
        print("=" * 92); print(f"{tag}: {base}  ->  {cond}"); print(q); print("=" * 92)
        print(s[["metric", "baseline", "conditioned", "improvement_pct", "ci_lo", "ci_hi",
                 "win_rate", "wilcoxon_p", "verdict"]].round(4).to_string(index=False)); print()
    STATS = pd.concat(STATS, ignore_index=True)
    STATS.to_csv(RESULTS_DIR / "tables/paired_stats.csv", index=False)
    print("win_rate = fraction of validation OBJECTS on which the conditioned model wins.")
    print("A CI spanning 0 is NOT resolved at this sample size; ci_hi < 0 is resolved as HARMFUL.")

## §15 · Visualisations

Numbers say which variant wins; these say *how*. Six views, each answering a question a table cannot:
what the completions look like, where the error sits, whether the colour palette is right, whether
material edges survive, how the object emerges over the reverse chain, and whether any of it holds up
as occlusion increases.

In [ ]:
def _view(ax, xyz, rgb, title="", s=4, lim=None):
    o = np.argsort(xyz[:, 1])
    ax.scatter(xyz[o, 0], xyz[o, 2], c=np.clip(rgb[o], 0, 1), s=s, linewidths=0)
    ax.set_aspect("equal"); ax.axis("off")
    if title: ax.set_title(title, fontsize=8)
    if lim: ax.set_xlim(lim[0]); ax.set_ylim(lim[1])


if MODELS_READY:
    n_show = min(4, len(VAL))
    ncol = 2 + len(TRAIN_MODELS)
    fig, axes = plt.subplots(n_show, ncol, figsize=(2.7 * ncol, 2.7 * n_show))
    axes = np.atleast_2d(axes)
    for r in range(n_show):
        g_gt = to_raw(X0_VAL[r]); vis = VIS_VAL[r].numpy()
        lim = ([g_gt[:, 0].min() - .1, g_gt[:, 0].max() + .1],
               [g_gt[:, 2].min() - .1, g_gt[:, 2].max() + .1])
        _view(axes[r, 0], g_gt[vis, :3], (g_gt[vis, 3:] + 1) / 2,
              f"input ({vis.mean():.0%} seen)" if r == 0 else "", lim=lim)
        for c, k in enumerate(TRAIN_MODELS):
            p = to_raw(SAMPLES[k][r])
            _view(axes[r, c + 1], p[:, :3], (p[:, 3:] + 1) / 2, k if r == 0 else "", lim=lim)
        _view(axes[r, -1], g_gt[:, :3], (g_gt[:, 3:] + 1) / 2, "ground truth" if r == 0 else "",
              lim=lim)
    fig.suptitle(f"Completions — {CATEGORY}, {DIFFICULTIES[0]}", fontsize=11)
    fig.tight_layout(); fig.savefig(RESULTS_DIR / "figures/15_completions.png", dpi=140,
                                    bbox_inches="tight"); plt.close(fig)
    print("saved figures/15_completions.png")

In [ ]:
# ---------------- where the error is: geometry vs colour, side by side ----------------
if MODELS_READY:
    r = 0
    g_gt = to_raw(X0_VAL[r]); vis = VIS_VAL[r].numpy()
    fig, axes = plt.subplots(2, len(TRAIN_MODELS), figsize=(3.1 * len(TRAIN_MODELS), 6.2))
    axes = np.atleast_2d(axes)
    for c, k in enumerate(TRAIN_MODELS):
        p = to_raw(SAMPLES[k][r])
        d_g2p, i_g2p = cKDTree(p[:, :3]).query(g_gt[:, :3], k=1)
        dE = ciede2000(srgb_to_lab((p[i_g2p, 3:] + 1) / 2), srgb_to_lab((g_gt[:, 3:] + 1) / 2))
        for row, (val, lab, cm) in enumerate([(d_g2p, "geometry error (dist to nearest pred)", "magma"),
                                              (dE, "colour error ($\\Delta E_{00}$)", "viridis")]):
            m = ~vis                                    # score only the invented points
            sc = axes[row, c].scatter(g_gt[m, 0], g_gt[m, 2], c=val[m], s=5, cmap=cm,
                                      vmin=0, vmax=np.percentile(val[m], 95))
            axes[row, c].scatter(g_gt[vis, 0], g_gt[vis, 2], c="0.85", s=3, zorder=0)
            axes[row, c].set_aspect("equal"); axes[row, c].axis("off")
            if row == 0: axes[row, c].set_title(k, fontsize=9)
            if c == len(TRAIN_MODELS) - 1:
                fig.colorbar(sc, ax=axes[row, :], shrink=.75, label=lab)
    fig.suptitle("Error maps — grey = given, coloured = invented", fontsize=11)
    fig.savefig(RESULTS_DIR / "figures/15_error_maps.png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    print("saved figures/15_error_maps.png")

In [ ]:
# ---------------- is the PALETTE right, independent of placement? ----------------
if MODELS_READY:
    miss_gt = np.concatenate([to_raw(X0_VAL[i])[~VIS_VAL[i].numpy(), 3:] for i in range(len(VAL))])
    lab_gt = srgb_to_lab((miss_gt + 1) / 2)
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
    names = ["L*  (lightness)", "a*  (green-red)", "b*  (blue-yellow)"]
    for ch in range(3):
        axes[ch].hist(lab_gt[:, ch], bins=60, density=True, color="0.25", alpha=.55,
                      label="ground truth")
        for k in TRAIN_MODELS:
            pm = np.concatenate([to_raw(SAMPLES[k][i])[~VIS_VAL[i].numpy(), 3:]
                                 for i in range(len(VAL))])
            axes[ch].hist(srgb_to_lab((pm + 1) / 2)[:, ch], bins=60, density=True,
                          histtype="step", lw=1.7, label=k)
        axes[ch].set_title(names[ch], fontsize=10); axes[ch].grid(alpha=.25)
        if ch == 0: axes[ch].legend(fontsize=8)
    fig.suptitle("Colour distribution over the missing region — no correspondence involved "
                 "(this is what colour_swd scores)", fontsize=10)
    fig.tight_layout(); fig.savefig(RESULTS_DIR / "figures/15_palette.png", dpi=130,
                                    bbox_inches="tight"); plt.close(fig)
    print("saved figures/15_palette.png")

In [ ]:
# ---------------- headline metrics with bootstrap intervals ----------------
if MODELS_READY:
    show = ["cd_l1", "fscore", "dE00_gt2pred", "colour_swd", "region_dE", "colour_edge_iou"]
    fig, axes = plt.subplots(2, 3, figsize=(13, 6.4))
    for ax, met in zip(axes.ravel(), show):
        vals, errs = [], []
        for k in TRAIN_MODELS:
            per_obj = MISS[MISS.method == k].groupby("obj")[met].mean().dropna().values
            rng = np.random.default_rng(0)
            bs = per_obj[rng.integers(0, len(per_obj), (2000, len(per_obj)))].mean(1)
            vals.append(per_obj.mean())
            errs.append([per_obj.mean() - np.percentile(bs, 2.5),
                         np.percentile(bs, 97.5) - per_obj.mean()])
        ax.bar(range(len(TRAIN_MODELS)), vals,
               yerr=np.array(errs).T, capsize=4,
               color=["#b0b0b0", "#6699cc", "#2f6f4f"][:len(TRAIN_MODELS)])
        ax.set_xticks(range(len(TRAIN_MODELS)))
        ax.set_xticklabels(TRAIN_MODELS, fontsize=8, rotation=12)
        arrow = "lower better" if METRIC_DIR[met] == "lower" else "higher better"
        ax.set_title(f"{met}  ({arrow})", fontsize=9); ax.grid(alpha=.25, axis="y")
    fig.suptitle("Missing region · mean over objects, 95% bootstrap CI", fontsize=11)
    fig.tight_layout(); fig.savefig(RESULTS_DIR / "figures/15_headline.png", dpi=130,
                                    bbox_inches="tight"); plt.close(fig)
    print("saved figures/15_headline.png")

In [ ]:
# ---------------- how the completion emerges over the reverse chain ----------------
if MODELS_READY:
    k = TRAIN_MODELS[-1]
    _, tr = sample(MODELS[k], X0_VAL[:1].to(DEV), VIS_VAL[:1].to(DEV),
                   seed=SEED, keep_trace=True, n_trace=7)
    tr = tr[:8]
    fig, axes = plt.subplots(1, len(tr) + 1, figsize=(2.3 * (len(tr) + 1), 2.6))
    for j, (t_, x) in enumerate(tr):
        p = to_raw(x[0]); _view(axes[j], p[:, :3], (p[:, 3:] + 1) / 2, f"t = {t_}", s=3)
    g = to_raw(X0_VAL[0]); _view(axes[-1], g[:, :3], (g[:, 3:] + 1) / 2, "ground truth", s=3)
    fig.suptitle(f"Reverse process, {k} — $\\hat{{x}}_0$ at decreasing noise "
                 "(geometry and colour denoise jointly)", fontsize=10)
    fig.tight_layout(); fig.savefig(RESULTS_DIR / "figures/15_reverse.png", dpi=130,
                                    bbox_inches="tight"); plt.close(fig)
    print("saved figures/15_reverse.png")

In [ ]:
# ---------------- does the ranking survive harder occlusion? ----------------
if MODELS_READY and len(DIFFICULTIES) > 1:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
    for ax, met in zip(axes, ["cd_l1", "dE00_gt2pred", "colour_swd"]):
        for k in TRAIN_MODELS:
            d = MISS[MISS.method == k].groupby("difficulty")[met].mean().reindex(DIFFICULTIES)
            ax.plot(range(len(d)), d.values, marker="o", lw=1.8, label=k)
        ax.set_xticks(range(len(DIFFICULTIES))); ax.set_xticklabels(DIFFICULTIES)
        ax.set_title(met, fontsize=10); ax.grid(alpha=.3)
    axes[0].legend(fontsize=8)
    fig.suptitle("Does the ordering hold as more of the object is hidden?", fontsize=11)
    fig.tight_layout(); fig.savefig(RESULTS_DIR / "figures/15_difficulty.png", dpi=130)
    plt.close(fig); print("saved figures/15_difficulty.png")
elif MODELS_READY:
    print(f"only one difficulty ({DIFFICULTIES[0]}) — set DIFFICULTIES = "
          '["simple","moderate","hard"] in §1 for the occlusion sweep.')

In [ ]:
# ---------------- interactive 3-D ----------------
if MODELS_READY:
    try:
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        r, k = 0, TRAIN_MODELS[-1]
        g = to_raw(X0_VAL[r]); p = to_raw(SAMPLES[k][r]); vis = VIS_VAL[r].numpy()
        def _c(a): return ["rgb(%d,%d,%d)" % tuple(np.clip((v + 1) / 2, 0, 1) * 255) for v in a]
        fig = make_subplots(rows=1, cols=3, specs=[[{"type": "scatter3d"}] * 3],
                            subplot_titles=[f"input ({vis.mean():.0%})", f"{k} completion",
                                            "ground truth"])
        for c, (xyz, col) in enumerate([(g[vis, :3], g[vis, 3:]), (p[:, :3], p[:, 3:]),
                                        (g[:, :3], g[:, 3:])]):
            fig.add_trace(go.Scatter3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
                                       marker=dict(size=1.6, color=_c(col)), showlegend=False),
                          row=1, col=c + 1)
        fig.update_layout(height=460, width=1250, title=f"{CATEGORY} · {VAL[r]['mid'][:16]}")
        out = RESULTS_DIR / "figures/15_interactive.html"
        fig.write_html(str(out), include_plotlyjs="cdn")
        print(f"saved {out.name}  (open it to rotate the clouds)")
    except Exception as e:
        print(f"plotly panel skipped: {e}")

## §16 · Report

In [ ]:
def md_table(df, index=False):
    """Markdown table with no hard dependency on `tabulate`."""
    try:
        return df.to_markdown(index=index)
    except Exception:
        d = df.reset_index() if index else df
        cols = [str(c) for c in d.columns]
        def fmt(v):
            if isinstance(v, float):
                return "nan" if v != v else f"{v:.4g}"
            return str(v)
        rows = ["| " + " | ".join(cols) + " |",
                "|" + "|".join("---" for _ in cols) + "|"]
        rows += ["| " + " | ".join(fmt(v) for v in rec) + " |" for rec in d.itertuples(index=False)]
        return "\n".join(rows)


def build_report():
    L = [f"# Joint 6-D Diffusion for Colored Point Cloud Completion — {CATEGORY}", "",
         "*Every number below was produced by this run.*", "",
         "## 1. Setup", "",
         f"- category **{CATEGORY}** (`{SYNSET}`), difficulties {DIFFICULTIES}",
         f"- {len(X0_TRAIN)} train / {len(X0_VAL)} val examples, {NUM_POINTS} points, disjoint by model id",
         f"- visible fraction: train {VIS_TRAIN.float().mean():.3f}, val {VIS_VAL.float().mean():.3f}",
         f"- normalisation from the **partial only** — no ground-truth frame leaks in (§4)",
         f"- {T_STEPS}-step {SCHEDULE} DDPM over all 6 channels; sampler {SAMPLER.upper()}"
         + (f" ({DDIM_STEPS} steps)" if SAMPLER == "ddim" else "")
         + (f", RePaint {REPAINT_JUMP}x{REPAINT_NJUMP}" if REPAINT else ""), ""]
    if (RESULTS_DIR / "tables/model_parameters.csv").exists():
        mp = pd.read_csv(RESULTS_DIR / "tables/model_parameters.csv")
        L += ["## 2. The ladder (parameter-matched)", "", md_table(mp), "",
              f"Spread {100*(mp['params'].max()-mp['params'].min())/mp['params'].min():.2f}% — "
              "so any difference below is architectural, not capacity. This closes the "
              "confound left open by the earlier study's +66.8% comparison.", ""]
    if not MODELS_READY:
        L += ["## Results", "", "Not trained. Run §11, set `RUN_FULL_EXPERIMENT = True` in §1, re-run §1."]
        return "\n".join(L)

    L += ["## 3. Main table — missing region", "", md_table(MAIN.round(4), index=True), ""]
    L += ["## 4. Paired comparisons", ""]
    for base, cond, tag, q in COMPARISONS:
        s = STATS[STATS.comparison == tag]
        if not len(s): continue
        L += [f"### {tag} · `{base}` → `{cond}`", "", f"*{q}*", "",
              s[["metric", "baseline", "conditioned", "improvement_pct", "ci_lo", "ci_hi",
                 "win_rate", "wilcoxon_p", "verdict"]].round(4).pipe(md_table), ""]
        helps = s[s.verdict == "helps"].metric.tolist()
        hurts = s[s.verdict == "hurts"].metric.tolist()
        L += [f"- resolved as helping: {', '.join(helps) if helps else '(none)'}",
              f"- resolved as harmful: {', '.join(hurts) if hurts else '(none)'}",
              f"- unresolved at this sample size: {len(s) - len(helps) - len(hurts)} of {len(s)}", ""]

    q1 = STATS[STATS.comparison == "Q1"]
    verdict = "did not replicate"
    if len(q1):
        n_help = (q1.verdict == "helps").sum()
        verdict = ("REPLICATED" if n_help >= len(q1) / 2 else
                   "did not replicate" if n_help == 0 else "partially replicated")
    L += ["## 5. What this establishes", "",
          f"The earlier denoising study found kNN-aggregated geometry conditioning worth **-63.2%** "
          f"eps-MSE at a 40/40 win rate, but with +66.8% more parameters. At matched parameter count "
          f"on the real completion task, that finding **{verdict}** (Q1).", "",
          "### Limits — read before quoting any number above", "",
          "1. **One training seed.** Every interval here is bootstrapped over validation *objects*, "
          "so it covers object-to-object variation and says nothing about run-to-run variation. Two "
          "architectures separated by a few percent could swap places under a different "
          f"initialisation. Until `SEED` is varied, treat a small effect as unmeasured, not absent.",
          "2. **No external baseline.** No PoinTr / SeedFormer / AdaPoinTr is run, so nothing here "
          "speaks to the state of the art — only to this ladder.",
          f"3. **Narrow slice.** One category ({CATEGORY}), difficulties {DIFFICULTIES}, "
          f"{len(X0_VAL)} validation objects.",
          "4. **Multiplicity.** " + f"{len(METRIC_DIR)} metrics x {len(COMPARISONS)} comparisons = "
          f"{len(METRIC_DIR)*len(COMPARISONS)} tests, uncorrected. A single metric crossing p<0.05 "
          "in isolation is expected by chance; look for agreement across the suite instead.",
          "5. **Tier-2 colour is partly a geometry metric.** Read `dE00_*` next to `match_rate`, and "
          "against the correspondence-free tier-3 numbers (`colour_swd`, `region_dE`), which cannot "
          "be moved by geometry error at all.", ""]
    return "\n".join(L)


rep = build_report()
(RESULTS_DIR / "report.md").write_text(rep)
print(rep[:3500])
print(f"\n... full report at {RESULTS_DIR/'report.md'} ({len(rep)} chars)")

In [ ]:
print("=" * 78); print("SAVED OUTPUTS"); print("=" * 78)
for sub in ["", "tables", "figures", "checkpoints", "history"]:
    d = RESULTS_DIR / sub
    fs = sorted(f for f in d.iterdir() if f.is_file()) if d.is_dir() else []
    if fs:
        print(f"\n{sub or 'report'}  ({d})")
        for f in fs:
            print(f"  {f.name:<44} {f.stat().st_size/1024:>9.1f} KB")

print("\n" + "=" * 78); print("NEXT STEP"); print("=" * 78)
if not MODELS_READY:
    print("Set RUN_FULL_EXPERIMENT = True in §1, re-run §1, then run from §13.")
else:
    print("""1. Widen the evidence before widening the model: DIFFICULTIES = ["simple","moderate","hard"]
   turns one number into a dose-response curve for the cost of sampling alone.
2. Repeat on car and chair. The pre-flight measured airplane 1.64 / car 1.39 / chair 0.96 on
   colour-boundary contrast, so chair is the honest stress test, not a formality.
3. Seeds 1 and 2. Current intervals cover variation across OBJECTS, not across training runs.
4. Only then: a published baseline (PoinTr / AdaPoinTr) on the same split. Until that exists,
   every claim here is internal to this ladder.""")